# Explore dataset structure

In [1]:
from dirs import *

from xflow.utils.io import scan_files
from xflow.utils.sql import merge_sqlite_dbs
from xflow import SqlProvider, PyTorchPipeline
from xflow.data import build_transforms_from_config

from config_utils import load_config, detect_machine
import pandas as pd
import sqlite3
from utils import *

experiment_name = "CLEAR_visualization"  
config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

c:\Users\qiyuanxu\Documents\GitHub\fiber-image-reconstruction-comparison
[config_utils] Using machine profile: win-qiyuanxu


# Load dataset

Different with training pipeline, data sample meta data is given

Mainly the run time sythetic dataset need to be compiled and generate here

In [2]:

# ============================
# Merge entire CLEAR 2025 dataset in to a single database
# ============================
db_paths = scan_files(dirs["processed_db_dir"], extensions=[".db"], return_type="str")
conn = merge_sqlite_dbs(db_paths, source_column="db_path")
sql = """
SELECT *
FROM mmf_dataset_metadata
"""
tables_df = pd.read_sql_query(sql, conn)
conn.close()
# optional: drop duplicate column names (e.g. both tables have "id", "db_path")
tables_df = tables_df.loc[:, ~tables_df.columns.duplicated()]
print(tables_df.shape)
tables_df.head()

(65561, 25)


,image_id,is_calibration,batch,purpose,image_path,comments,beam_settings,create_time,is_saturated_ground_truth,is_saturated_fiber_output,...,other_config,dataset,gaussian_beam_params,moments_beam_params,coupling_ratio,ground_truth_stats,fiber_output_stats,original_image_crop_size,original_image_crop_coords,db_path
0,1763715583307716800,0,3,testset,dataset/1763715583211057600.png,real beam time; 10Hz repetition rate; 6 bunches,"{""CLEAR_magnets"": {""QFD0880"": null, ""QDD0515"":...",2025-11-21 08:59:43,0,0,...,"{'dmd_config': {'type': 'DummyDMD', 'status': ...",2025-11-21-morning,None,"[0.5116365863652134, 0.4971741521602921, 0.068...",60.955012,"{'max': 130, 'min': 0, 'mean': 0.4656829833984...","{'max': 132, 'min': 0, 'mean': 28.385711669921...","(520, 520)","((718, 349), (1238, 869))",C:/Users/qiyuanxu/Documents/DataHub/processed/...
1,1763715583883483500,0,3,testset,dataset/1763715583799784100.png,real beam time; 10Hz repetition rate; 6 bunches,"{""CLEAR_magnets"": {""QFD0880"": null, ""QDD0515"":...",2025-11-21 08:59:43,0,0,...,"{'dmd_config': {'type': 'DummyDMD', 'status': ...",2025-11-21-morning,None,"[0.511181665142949, 0.4976276694920029, 0.0701...",60.819639,"{'max': 129, 'min': 0, 'mean': 0.4692840576171...","{'max': 133, 'min': 0, 'mean': 28.541687011718...","(520, 520)","((718, 349), (1238, 869))",C:/Users/qiyuanxu/Documents/DataHub/processed/...
2,1763715584475775400,0,3,testset,dataset/1763715584385935400.png,real beam time; 10Hz repetition rate; 6 bunches,"{""CLEAR_magnets"": {""QFD0880"": null, ""QDD0515"":...",2025-11-21 08:59:44,0,0,...,"{'dmd_config': {'type': 'DummyDMD', 'status': ...",2025-11-21-morning,None,"[0.5109162041613701, 0.49774370018139635, 0.07...",61.413787,"{'max': 119, 'min': 0, 'mean': 0.4261016845703...","{'max': 122, 'min': 0, 'mean': 26.168518066406...","(520, 520)","((718, 349), (1238, 869))",C:/Users/qiyuanxu/Documents/DataHub/processed/...
3,1763715585074460800,0,3,testset,dataset/1763715584975413500.png,real beam time; 10Hz repetition rate; 6 bunches,"{""CLEAR_magnets"": {""QFD0880"": null, ""QDD0515"":...",2025-11-21 08:59:45,0,0,...,"{'dmd_config': {'type': 'DummyDMD', 'status': ...",2025-11-21-morning,None,"[0.5114130814784044, 0.497440066189149, 0.0698...",60.906474,"{'max': 131, 'min': 0, 'mean': 0.4659576416015...","{'max': 132, 'min': 0, 'mean': 28.379837036132...","(520, 520)","((718, 349), (1238, 869))",C:/Users/qiyuanxu/Documents/DataHub/processed/...
4,1763715585737017200,0,3,testset,dataset/1763715585587687400.png,real beam time; 10Hz repetition rate; 6 bunches,"{""CLEAR_magnets"": {""QFD0880"": null, ""QDD0515"":...",2025-11-21 08:59:45,0,0,...,"{'dmd_config': {'type': 'DummyDMD', 'status': ...",2025-11-21-morning,None,"[0.511817280305108, 0.49704200823657024, 0.071...",61.048780,"{'max': 129, 'min': 0, 'mean': 0.4529418945312...","{'max': 129, 'min': 0, 'mean': 27.651550292968...","(520, 520)","((718, 349), (1238, 869))",C:/Users/qiyuanxu/Documents/DataHub/processed/...


In [ ]:
# pip install xflow-py
from xflow.extensions.physics.pipeline import CachedBasisPipeline, IndexCombinator
from xflow.extensions.physics import pattern_gen
from tqdm import tqdm

"""
    Contract:
    1. Unify the data format for visualization like (2, 1000, 1, 256, 256) (fiber_origin_image_type, num_samples, C, H, W)
    2. Add labels or class to data samples with index based order matter
    3. Union data samples and labels into two single iterable for down stream visualization (order still matter)
"""

# ============================
# DMD + orthognal basis syth data augmentation
# ============================
dmd_orth_basis_provider = SqlProvider(
    sources={"connection": dirs["DMD_only"]["dataset_db_dir"], "sql": config["sql"]["train"]}, output_config={'list': "image_path"}
)

dmd_validation_provider = SqlProvider(
    sources={"connection": dirs["DMD_only"]["dataset_db_dir"], "sql": config["sql"]["eval"]}, output_config={'list': "image_path"}
)

config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": dirs["DMD_only"]["dataset_extracted_dir"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

val_pipeline = PyTorchPipeline(
    dmd_validation_provider, 
    transforms[:-1],
).to_numpy() 

# ========== SGM simulation pattern generator ==========
canvas = pattern_gen.DynamicPatterns(*config["simulation"]["canvas_size"])
canvas.set_postprocess_fns(build_transforms_from_config(config["simulation"]["process_functions"]))
canvas._distributions = [pattern_gen.StaticGaussianDistribution(canvas) for _ in range(config["simulation"]["total_Guassian_num"])]
canvas.set_threshold(config["simulation"]["minimum_pixel_threshold"])
stream = canvas.pattern_stream(std_1=config["simulation"]["std_1"], std_2=config["simulation"]["std_2"],
                               max_intensity=config["simulation"]["max_intensity"], fade_rate=config["simulation"]["fade_rate"], 
                               distribution=config["simulation"]["distribution"]) 

# ======== random combinator using index + SGM ========
combinator = IndexCombinator(
    pattern_provider=stream,
    transforms= build_transforms_from_config(config["combinator"]["transforms"]["torch"]),
)
train_dataset = CachedBasisPipeline(
    dmd_orth_basis_provider, 
    combinator=combinator, 
    transforms=transforms, 
    num_samples=config["data"]["total_train_samples"], 
    seed=config["seed"],
    eager=True
)

samples = []
for _ in tqdm(range(1000)):
    samples.append(next(iter(train_dataset)))
samples = np.stack(samples, axis=0)         # (1000, 2, 1, 256, 256)
samples = np.transpose(samples, (1, 0, 2, 3, 4))  # (2, 1000, 1, 256, 256)

labels = ["DMD_syth (training set)"] * len(samples[0]) + ["DMD_val (test set)"] * len(val_pipeline[0])
result = np.concatenate([samples, np.stack(val_pipeline, axis=0)], axis=1) 



# Non syth data just use the master database table, since no run time requirement
# ============================
# Chromox screen set with subsample
# ============================
chromox_provider = SqlProvider(
    sources={"connection": dirs["DMD_only"]["dataset_db_dir"], "sql": config["sql"]["train"]}, output_config={'list': "image_path", "label": "dataset"}
)

labels = 




# ============================
# Yag screen set with subsample
# ============================







# ============================
# Laser scan (Chromox) set with subsample
# ============================






# ============================
# Laser scan (Yag) set with subsample
# ============================





# Latent space distribution

In [ ]:
import numpy as np
import imageio.v2 as imageio
from pathlib import Path
from xflow.utils.visualization import DimReducer, Embedding3DPlot

def embed_images(images, final_method="umap", final_dim=2, random_state=42):
    X = np.asarray(images, dtype=np.float32)
    X = X.reshape(X.shape[0], -1)
    X /= (X.max() + 1e-12)

    pca = DimReducer("pca", n_components=min(50, X.shape[1]), random_state=random_state)
    X50 = pca.fit_transform(X)

    method_norm = "".join(ch for ch in str(final_method).lower() if ch.isalnum())  # "t-SNE" -> "tsne"
    if method_norm == "tsne":
        final = DimReducer("tsne", n_components=final_dim, random_state=random_state, init="pca")
    elif method_norm == "umap":
        final = DimReducer("umap", n_components=final_dim, random_state=random_state)
    else:
        raise ValueError(f"Unsupported final_method: {final_method}")

    Z = final.fit_transform(X50)
    return Z, pca, final

# 1) Prepare PCA, t-SNE, UMAP coords
method = "t-SNE"
index = 0  # 0 for fiber output speckle, 1 for original image
images = result[index]  # (N, 1, 256, 256)

title = "{} projection - MMF output speckle".format(method.upper()) if index == 0 else "{} projection - Original images".format(method.upper())
coords, _, _ = embed_images(images, final_method=method, final_dim=3, random_state=42)

# 2) Configure plotter once (object-level defaults)
plotter = Embedding3DPlot(
    coords=coords,
    labels=labels,
    title=title,
    point_size=4,

    # Shared defaults for both Plotly projections and Matplotlib frames
    show_projections=False,      # set True if you want wall projection points
    projection_envelope=True,    # shading on wall distributions
    projection_alpha=0.25,
    projection_size_scale=0.7,
    projection_gap_ratio=0.06,
    projection_envelope_alpha=0.18,
)
# plotter.show(elev=25, azim=35); 

# 3) Save orbit GIF from Matplotlib frames
out_dir = Path("results/gif")
out_dir.mkdir(parents=True, exist_ok=True)

frames = []
n_frames = 120
base_elev, base_azim = 25.0, 40.0
azim_radius = 25.0 / 2.0
elev_radius = 10.0 / 2.0
t = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

for tt in t:
    azim = base_azim + azim_radius * np.cos(tt)
    elev = base_elev + elev_radius * np.sin(tt)

    frame = plotter.get_matplotlib_frame(
        elev=float(elev),
        azim=float(azim),
        dpi=120,
        # optional per-call override:
        # show_projections=True,
        # projection_envelope=False,
    )
    frames.append(frame)

imageio.mimsave(out_dir / "embedding_orbit_loop.gif", frames, fps=50, loop=0)

# Fiber coupling rate vs time

In [ ]:
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

# Minimal inputs from your dataframe
timestamps = tables_df["image_id"].astype(np.int64).tolist()
coupling_ratio = tables_df["coupling_ratio"].astype(float).tolist()
classes = tables_df["image_device"].tolist()

def plot_coupling_time_series(timestamps, coupling_ratio, classes, dot_size=50):
    time_labels = [datetime.fromtimestamp(ts / 1e9).strftime("%Y-%m-%d %H:%M") for ts in timestamps]
    x_indices = list(range(len(coupling_ratio)))

    plt.figure(figsize=(10, 5))

    # 3rd-degree polynomial trend
    z = np.polyfit(x_indices, coupling_ratio, 3)
    p = np.poly1d(z)
    # plt.plot(x_indices, p(x_indices), color="red", dashes=(5, 5), linewidth=1, label="Overall Trend", zorder=3)

    custom_colors = ["C1", "C2", "C0"]
    unique_classes = sorted(set(classes))

    for i, cls in enumerate(unique_classes):
        cls_x = [x_indices[j] for j, c in enumerate(classes) if c == cls]
        cls_y = [coupling_ratio[j] for j, c in enumerate(classes) if c == cls]
        plt.scatter(cls_x, cls_y, label=cls, s=dot_size, color=custom_colors[i % len(custom_colors)], zorder=2)

    step = max(1, len(x_indices) // 10)
    tick_positions = x_indices[::step]
    plt.xticks(tick_positions, [time_labels[i] for i in tick_positions], rotation=15, ha="right")

    plt.xlabel("Time (Sequential)")
    plt.ylabel("Coupling Ratio")
    plt.title("Coupling Ratio vs Time")
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()

plot_coupling_time_series(timestamps, coupling_ratio, classes, dot_size=1)

# Image file size change vs time

In [ ]:
sql = """
SELECT
    d.*,
    c.experiment_description,
    c.image_source,
    c.image_device,
    c.fiber_config,
    c.camera_config,
    c.other_config
FROM mmf_dataset_metadata AS d
LEFT JOIN mmf_experiment_config AS c
  ON c.id = d.config_id
 AND c.db_path = d.db_path;
"""

with sqlite3.connect(str(dirs["merged_db_path"])) as con:
    tables_df = pd.read_sql_query(sql, con)

In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime


temp = [
    (Path(db).parent.parent / img).as_posix()
    for db, img in zip(tables_df["db_path"], tables_df["image_path"])
]

sizes_kb = [os.path.getsize(path) / 1024 for path in temp]
timestamps = tables_df['image_id'].astype(int).tolist()  # Assuming image_id is a timestamp in seconds
classes = tables_df["image_device"].tolist()  # Assuming "image_device" is the class label

def plot_compressed_time_series(timestamps, sizes_kb, classes, dot_size=50):
    time_labels = [datetime.fromtimestamp(ts / 1e9).strftime('%Y-%m-%d %H:%M') for ts in timestamps]
    x_indices = list(range(len(sizes_kb)))
    
    plt.figure(figsize=(10, 5))
    
    # Fit a 3rd-degree polynomial trend curve across all data points
    z = np.polyfit(x_indices, sizes_kb, 3)
    p = np.poly1d(z)
    plt.plot(x_indices, p(x_indices), color='red', dashes=(5, 5), linewidth=1, label='Overall Trend', zorder=3)
    
    # Define specific colors and sort classes for consistent assignment
    custom_colors = ['C1', 'C2', 'C0']
    unique_classes = sorted(list(set(classes)))
    
    for i, cls in enumerate(unique_classes):
        cls_x = [x_indices[j] for j, c in enumerate(classes) if c == cls]
        cls_y = [sizes_kb[j] for j, c in enumerate(classes) if c == cls]
        
        # Select color, wrapping around if there are more than 3 classes
        color = custom_colors[i % len(custom_colors)]
        
        plt.scatter(cls_x, cls_y, label=cls, s=dot_size, color=color, zorder=2)
    
    # Sample the X-axis ticks
    step = max(1, len(x_indices) // 10)
    tick_positions = x_indices[::step]
    selected_labels = [time_labels[i] for i in tick_positions]
    
    # Apply manual rotation and alignment
    plt.xticks(tick_positions, selected_labels, rotation=15, ha='right')
    
    plt.xlabel('Time (Sequential)')
    plt.ylabel('File Size (KB)')
    plt.title('File Size vs Time')
    
    plt.legend() 
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    
    plt.show()

# Example call:
plot_compressed_time_series(timestamps, sizes_kb, classes, dot_size=1)

# Check Abnormal samples

## Coupling ratio too big or too small samples plot

## Staturation